# 第 3 週 實作｜導數的應用 (一)（Colab notebook）

**目標**：把理論課的三個重點**親手跑出來、畫出來**——
1. 牛頓法 (Newton's Method) 求根 + 觀察二次收斂,並和第 1 週的二分法比速度
2. 線性近似 (Linear Approximation) 誤差視覺化(函數 vs 切線,放大看誤差)
3. 羅必達法則 (L'Hôpital's Rule) 的圖像直覺(兩函數比值 → 兩導數比值)

**用法**：上傳到 [Google Colab](https://colab.research.google.com/) 或本機 Jupyter,由上往下逐格執行。
標「`# TODO 學生練習`」的格子留給學生填。

> 圖表標籤用英文/數學符號以避免中文變豆腐字;中文都放在說明格。

In [ ]:
# === 環境設定(先跑這格)===
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
print("環境就緒, numpy", np.__version__)

### (選用)讓圖表顯示中文

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次),之後的圖就能顯示中文。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("預設英文標籤;要中文請見上一格說明")

## Lab 1｜牛頓法求根 × 二次收斂

**牛頓法**:在 $x_n$ 作切線,滑到 $x$ 軸得下一個猜測 $x_{n+1}=x_n-\dfrac{f(x_n)}{f'(x_n)}$。
求 $\sqrt2$ 就是解 $x^2-2=0$。看誤差欄:每一步的誤差大約是前一步的**平方**——這就是**二次收斂**。

In [ ]:
# 牛頓法:沿切線滑到 x 軸  x_{n+1} = x_n - f(x_n)/f'(x_n)
def newton(f, df, x0, tol=1e-13, maxit=20):
    """回傳每一步的 x 值序列。"""
    xs = [x0]
    x = x0
    for _ in range(maxit):
        x = x - f(x) / df(x)
        xs.append(x)
        if abs(f(x)) < tol:
            break
    return xs

# 解 x^2 - 2 = 0,即求 sqrt(2)
f  = lambda x: x**2 - 2
df = lambda x: 2 * x
xs = newton(f, df, 1.0)
root = math.sqrt(2)
print("step   x_n                  error |x_n - sqrt2|")
for n, xn in enumerate(xs):
    print(f"{n:>3}    {xn:.15f}    {abs(xn - root):.2e}")

和第 1 週的**二分法 (bisection)** 比一比。二分法每步只把區間砍一半(線性收斂),
牛頓法正確位數大約每步加倍(二次收斂)——差距非常懸殊。

In [ ]:
# 和第 1 週的二分法比速度:牛頓法(二次收斂)vs 二分法(線性收斂)
def bisect(f, a, b, tol=1e-13, maxit=200):
    steps = 0
    while (b - a) / 2 > tol and steps < maxit:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
        steps += 1
    return (a + b) / 2, steps

xs = newton(f, df, 1.0)
xb, kb = bisect(f, 1.0, 2.0)
print(f"Newton    : {len(xs)-1:>3} 次迭代 -> x = {xs[-1]:.15f}")
print(f"Bisection : {kb:>3} 次迭代 -> x = {xb:.15f}")
print("牛頓法正確位數大約每步加倍(二次),二分法每步只砍一半(線性)。")

把兩者的誤差畫在對數座標上:牛頓法幾乎是「懸崖式」陡降,二分法是一條穩定下滑的直線。

In [ ]:
# 收斂速度視覺化(誤差取對數):牛頓法陡降,二分法直線下滑
xs = newton(f, df, 1.0)
nerr = [max(abs(x - root), 1e-18) for x in xs]

a, b = 1.0, 2.0
berr = []
for _ in range(45):
    m = (a + b) / 2
    if f(a) * f(m) <= 0:
        b = m
    else:
        a = m
    berr.append(max(abs((a + b) / 2 - root), 1e-18))

plt.semilogy(range(len(nerr)), nerr, 'o-', label='Newton (quadratic)')
plt.semilogy(range(1, len(berr) + 1), berr, 's--', label='Bisection (linear)')
plt.xlabel('iteration'); plt.ylabel('error (log scale)')
plt.title('Newton vs Bisection: convergence speed')
plt.legend(); plt.show()

In [ ]:
# TODO 學生練習:用牛頓法求 x^3 - 2 = 0 的根(2 的立方根),x0 = 1
# g  = lambda x: x**3 - 2
# dg = lambda x: 3*x**2
# xs = newton(g, dg, 1.0)
# for n, xn in enumerate(xs): print(n, f"{xn:.12f}")
# 參考答案:收斂到 1.259921049...

## Lab 2｜線性近似誤差視覺化

$f(x)=\sqrt{x}$ 在 $a=4$ 的切線是 $L(x)=2+0.25(x-4)$。
先看整體,再**放大 $a$ 附近**——切線幾乎貼著曲線;最後把誤差 $|f-L|$ 畫出來,
你會看到它離 $a$ 越遠才慢慢張開(誤差約和 $(x-a)^2$ 成正比)。

In [ ]:
# 線性近似:f(x)=sqrt(x) 在 a=4 的切線 L(x)=2+0.25(x-4)
f = lambda x: np.sqrt(x)
a = 4.0
L = lambda x: 2 + 0.25 * (x - a)

x = np.linspace(1, 9, 400)
plt.plot(x, f(x), label='f(x) = sqrt(x)')
plt.plot(x, L(x), '--', label='L(x) = 2 + 0.25(x-4)')
plt.plot(a, f(a), 'o', color='k')
plt.title('Linear approximation of sqrt(x) at a=4')
plt.legend(); plt.show()

print("sqrt(4.1) approx L(4.1) =", round(L(4.1), 6), "  actual =", round(math.sqrt(4.1), 6))

In [ ]:
# 放大 a 附近:切線幾乎貼著曲線;再畫誤差 |f-L| 隨遠離 a 而變大
x = np.linspace(3.5, 4.5, 400)
plt.plot(x, f(x), label='f(x)')
plt.plot(x, L(x), '--', label='L(x)')
plt.plot(a, f(a), 'o', color='k')
plt.title('Zoom near a=4: tangent hugs the curve')
plt.legend(); plt.show()

x = np.linspace(2, 6, 400)
plt.plot(x, np.abs(f(x) - L(x)), color='C3', label='|f(x) - L(x)|')
plt.axvline(a, color='gray', ls=':')
plt.title('Error grows like (x-a)^2, tiny near a')
plt.legend(); plt.show()

## Lab 3｜羅必達法則的圖像直覺

$\displaystyle\lim_{x\to0}\dfrac{\sin x}{x}$ 是 $\tfrac00$。羅必達說它等於**兩導數的比值** $\dfrac{(\sin x)'}{(x)'}\Big|_{0}=\dfrac{\cos0}{1}=1$。
畫出來:比值 $f/g$ 在 $0$ 附近確實壓向 $1$。為什麼?因為靠近 $0$ 時每個函數都**幾乎等於自己的切線**,
$\sin x\approx x$,於是比值 $\approx$ 切線斜率之比 $=f'(0)/g'(0)$。

In [ ]:
# 羅必達直覺:lim sin(x)/x 在 x->0
# 兩函數比值(f/g) 趨近 兩導數比值 f'(0)/g'(0) = cos(0)/1 = 1
f = lambda x: np.sin(x)
g = lambda x: x
x = np.linspace(-3, 3, 601); x = x[x != 0]
plt.plot(x, f(x) / g(x), label='f/g = sin(x)/x')
plt.axhline(1, color='C1', ls='--', label="deriv ratio f'(0)/g'(0) = 1")
plt.title("L'Hopital intuition: sin(x)/x -> 1 as x -> 0")
plt.legend(); plt.ylim(0, 1.2); plt.show()

In [ ]:
# 為什麼?靠近 0 時每個函數 ~ 它的切線:sin(x) ~ x,x ~ x
# 所以 f/g ~ (切線斜率比) = f'(0)/g'(0)
x = np.linspace(-1, 1, 400)
plt.plot(x, np.sin(x), label='f(x) = sin(x)')
plt.plot(x, x, '--', label="tangent y = f'(0) x = x")
plt.axhline(0, color='gray', lw=0.8); plt.axvline(0, color='gray', lw=0.8)
plt.title('Near 0, sin(x) hugs its tangent line y = x')
plt.legend(); plt.show()

# sympy 驗證
xs = sp.symbols('x')
print("sympy: lim sin(x)/x =", sp.limit(sp.sin(xs)/xs, xs, 0))

## 收尾 · 與筆試的連結

| 這格實作 | 對應觀念 |
|---|---|
| Lab 1 牛頓法 × 收斂 | Newton's Method(觀念 13);對照 W1 二分法 |
| Lab 2 線性近似誤差 | Linear Approximation / Differentials(觀念 3、4） |
| Lab 3 羅必達直覺 | L'Hôpital's Rule(觀念 7);切線近似(觀念 3) |

### 進階徽章(選做)
1. 把 Lab 1 的牛頓法改求 $\sqrt[3]{2}$($f(x)=x^3-2$),數數看幾步收斂。
2. 讓牛頓法印出 $|e_{n+1}|/e_n^2$,看它是否趨近一個常數(二次收斂的證據）。
3. 用 `sympy`:`sp.limit(x*sp.log(x), x, 0, '+')` 驗證 Lab / 例題裡的 $0\cdot\infty$ 極限。